# Challenge 1 - T-test

In statistics, t-test is used to test if two data samples have a significant difference between their means. There are two types of t-test:

* **Student's t-test** (a.k.a. independent or uncorrelated t-test). This type of t-test is to compare the samples of **two independent populations** (e.g. test scores of students in two different classes). `scipy` provides the [`ttest_ind`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_ind.html) method to conduct student's t-test.

* **Paired t-test** (a.k.a. dependent or correlated t-test). This type of t-test is to compare the samples of **the same population** (e.g. scores of different tests of students in the same class). `scipy` provides the [`ttest_re`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_rel.html) method to conduct paired t-test.

Both types of t-tests return a number which is called the **p-value**. If p-value is below 0.05, we can confidently declare the null-hypothesis is rejected and the difference is significant. If p-value is between 0.05 and 0.1, we may also declare the null-hypothesis is rejected but we are not highly confident. If p-value is above 0.1 we do not reject the null-hypothesis.

Read more about the t-test in [this article](https://researchbasics.education.uconn.edu/t-test/) and [this Quora](https://www.quora.com/What-is-the-difference-between-a-paired-and-unpaired-t-test). Make sure you understand when to use which type of t-test. 

In [3]:
# Import libraries

import pandas as pd
import scipy.stats as stats

#### Import dataset

In this challenge we will work on the Pokemon dataset you have used last week. The goal is to test whether different groups of pokemon (e.g. Legendary vs Normal, Generation 1 vs 2, single-type vs dual-type) have different stats (e.g. HP, Attack, Defense, etc.).

In [4]:
# Import dataset

pokemon = pd.read_csv('../data/Pokemon.csv')

pokemon.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [5]:
pokemon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 13 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   #           800 non-null    int64 
 1   Name        800 non-null    object
 2   Type 1      800 non-null    object
 3   Type 2      414 non-null    object
 4   Total       800 non-null    int64 
 5   HP          800 non-null    int64 
 6   Attack      800 non-null    int64 
 7   Defense     800 non-null    int64 
 8   Sp. Atk     800 non-null    int64 
 9   Sp. Def     800 non-null    int64 
 10  Speed       800 non-null    int64 
 11  Generation  800 non-null    int64 
 12  Legendary   800 non-null    bool  
dtypes: bool(1), int64(9), object(3)
memory usage: 75.9+ KB


In [6]:
pokemon.isna().sum()

#               0
Name            0
Type 1          0
Type 2        386
Total           0
HP              0
Attack          0
Defense         0
Sp. Atk         0
Sp. Def         0
Speed           0
Generation      0
Legendary       0
dtype: int64

In [7]:
pokemon.loc[pokemon['Type 2'].isna()]

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False
5,5,Charmeleon,Fire,NaN,405,58,64,58,80,65,80,1,False
9,7,Squirtle,Water,NaN,314,44,48,65,50,64,43,1,False
10,8,Wartortle,Water,NaN,405,59,63,80,65,80,58,1,False
11,9,Blastoise,Water,NaN,530,79,83,100,85,105,78,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
775,705,Sliggoo,Dragon,NaN,452,68,75,53,83,113,60,6,False
776,706,Goodra,Dragon,NaN,600,90,100,70,110,150,80,6,False
788,712,Bergmite,Ice,NaN,304,55,69,85,32,35,28,6,False
789,713,Avalugg,Ice,NaN,514,95,117,184,44,46,28,6,False


There is only one column with NaN values, Type 2. I am not a domain expert, but I presume that these are valid types of pokemon that simply occur only in one type. 

#### First we want to define a function with which we can test the means of a feature set of two samples. 

In the next cell you'll see the annotations of the Python function that explains what this function does and its arguments and returned value. This type of annotation is called **docstring** which is a convention used among Python developers. The docstring convention allows developers to write consistent tech documentations for their codes so that others can read. It also allows some websites to automatically parse the docstrings and display user-friendly documentations.

Follow the specifications of the docstring and complete the function.

In [ ]:
# I extended the function to automatically decide whether to set the equal_var switch, i.e. whether to use Student's or Welch's t-test
def t_test_features(s1, s2, features=['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed', 'Total'],
                    auto_equalvar=True,levene_alpha=0.05, size_diff_cutoff=0.05):
    """Test means of a feature set of two samples
    
    Args:
        s1 (dataframe): sample 1
        s2 (dataframe): sample 2
        features (list): an array of features to test
        auto_equalvar: run an automatic test based on Levene statistic to evaluate whether to use Welch's or Student's t-test
        levene_alpha: threshold for Levene statistic to assume significant difference in variance
        size_diff_cutoff: maximum difference ratio in size between sample sizes before switching to Welch's t-test
    
    Returns:
        dict: a dictionary of t-test scores for each feature where the feature name is the key and the p-value is the value
    """
    results = {}

    # Your code here

    for f in features:
        # test if we should assume equal variance or not to use appropriate t-test
        # using Levene test to test for variance difference and additionally checking for difference in sample size
        if auto_equalvar:
            _, p_levene = stats.levene(s1[f].dropna(), s2[f].dropna())
            # Check size imbalance
            size_ratio = abs(len(s1) - len(s2)) / max(len(s1), len(s2))
            # Combine both tests
            equalvar = not (p_levene < levene_alpha or size_ratio > size_diff_cutoff)        
        else:
            equalvar = False  # use Welch by default
            
        res = stats.ttest_ind(s1[f],s2[f],equal_var=equalvar)
        results[f] = {
            'pval': float(res.pvalue),
            'equal_var': equalvar,
            'p_levene': float(p_levene) if auto_equalvar else None
        }
    
    return results

#### Using the `t_test_features` function, conduct t-test for Lengendary vs non-Legendary pokemons.

*Hint: your output should look like below:*

```
{'HP': 1.0026911708035284e-13,
 'Attack': 2.520372449236646e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239812e-21,
 'Sp. Def': 2.2949327864052826e-15,
 'Speed': 1.049016311882451e-18,
 'Total': 9.357954335957446e-47}
 ```

In [22]:
# Your code here
# subsetting the dataframe
legendary = pokemon.loc[pokemon['Legendary']]
ordinary = pokemon.loc[~(pokemon['Legendary'])]


print(f"n of ordinary: {ordinary.size}")
print(f"n of legendary: {legendary.size}")

# calling the t-test battle
# importantly, we do not assume equal variance for both groups 
# otherwise we don't get the correct results ;P 
# but also it may make sense to assume that these groups vary in different ways
# not least because of the very different size of both groups
t_test_features(ordinary,legendary)

n of ordinary: 9555
n of legendary: 845


{'HP': {'pval': 1.0026911708035284e-13,
  'equal_var': False,
  'p_levene': 0.5758428070138398},
 'Attack': {'pval': 2.520372449236646e-16,
  'equal_var': False,
  'p_levene': 0.9944645877795636},
 'Defense': {'pval': 4.8269984949193316e-11,
  'equal_var': False,
  'p_levene': 0.3203429011757649},
 'Sp. Atk': {'pval': 1.5514614112239812e-21,
  'equal_var': False,
  'p_levene': 0.37014131900583735},
 'Sp. Def': {'pval': 2.2949327864052826e-15,
  'equal_var': False,
  'p_levene': 0.7764108095205825},
 'Speed': {'pval': 1.049016311882451e-18,
  'equal_var': False,
  'p_levene': 0.0017862391470896439},
 'Total': {'pval': 9.357954335957446e-47,
  'equal_var': False,
  'p_levene': 9.640747382375356e-09}}

#### From the test results above, what conclusion can you make? Do Legendary and non-Legendary pokemons have significantly different stats on each feature?

In [ ]:
# Your comment here

The p-value for each of the tested stats is well below 0.001, providing ample evidence against the null hypothesis that legendary and ordinary  pokemon show no significant difference in their average values for each feature.
While not indicated above, our alpha has typically been set at 0.05, so the threshold for rejecting the null hypothesis is met, but the results would even remain significant under stricter thresholds ($\alpha < 0.01$ or $\alpha < 0.001$)

#### Next, conduct t-test for Generation 1 and Generation 2 pokemons.

In [23]:
# Your code here

# subsetting
gen1 = pokemon.loc[pokemon['Generation'] == 1]
gen2 = pokemon.loc[pokemon['Generation'] == 2]

print(f"n of gen1: {gen1.size}")
print(f"n of gen2: {gen2.size}")

# testing
# setting equalvar to False again: while group sizes don't differ as dramatically as before
# one group still has almost double the size of the other
t_test_features(gen1,gen2)



n of gen1: 2158
n of gen2: 1378


{'HP': {'pval': 0.14551697834219623,
  'equal_var': False,
  'p_levene': 0.7522675628722661},
 'Attack': {'pval': 0.24721958967217725,
  'equal_var': False,
  'p_levene': 0.686026571819782},
 'Defense': {'pval': 0.5677711011725426,
  'equal_var': False,
  'p_levene': 0.028161170529703957},
 'Sp. Atk': {'pval': 0.12332165977104388,
  'equal_var': False,
  'p_levene': 0.06813966540549882},
 'Sp. Def': {'pval': 0.18829872292645752,
  'equal_var': False,
  'p_levene': 0.3407655444327107},
 'Speed': {'pval': 0.00239265937312135,
  'equal_var': False,
  'p_levene': 0.48868555797283},
 'Total': {'pval': 0.5631377907941676,
  'equal_var': False,
  'p_levene': 0.8876679590679554}}

#### What conclusions can you make?

In [ ]:
# Your comment here

There is a significant difference between Gen 1 and Gen 2 Pokemon in the feature `Speed` at the level of $p < 0.05$. No statistically significant difference could be detected for any other tested feature.

#### Compare pokemons who have single type vs those having two types.

In [24]:
# Your code here

# subsetting
singletype = pokemon.loc[pokemon['Type 2'].isna()]
doubletype = pokemon.loc[~pokemon['Type 2'].isna()]

print(f"n of pokemons with one type: {singletype.size}")
print(f"n of pokemons with two types: {doubletype.size}")

# testing
# setting equalvar to False again: while group sizes don't differ as dramatically as before
# one group still has almost double the size of the other
t_test_features(singletype,doubletype)


n of pokemons with one type: 5018
n of pokemons with two types: 5382


{'HP': {'pval': 0.11314389855379413,
  'equal_var': False,
  'p_levene': 0.3584145973463879},
 'Attack': {'pval': 0.00014932578145948305,
  'equal_var': False,
  'p_levene': 0.04064876885416534},
 'Defense': {'pval': 2.7978540411514693e-08,
  'equal_var': False,
  'p_levene': 0.0032300463879700514},
 'Sp. Atk': {'pval': 0.00013876216585667907,
  'equal_var': False,
  'p_levene': 0.03492968137489321},
 'Sp. Def': {'pval': 0.00010730610934512779,
  'equal_var': False,
  'p_levene': 0.3513164201878122},
 'Speed': {'pval': 0.02421703281819093,
  'equal_var': False,
  'p_levene': 0.7286905369996797},
 'Total': {'pval': 1.1157056505229961e-07,
  'equal_var': False,
  'p_levene': 0.7976089324555451}}

#### What conclusions can you make?

In [ ]:
# Your comment here

These two groups show a significant difference in their average Attack, Defense, Sp.Atk, Sp.Def, Speed and Total stats at p smaller than 0.05. No significant difference can be found for HP.

#### Now, we want to compare whether there are significant differences of `Attack` vs `Defense`  and  `Sp. Atk` vs `Sp. Def` of all pokemons. Please write your code below.

*Hint: are you comparing different populations or the same population?*

In [27]:
# Your code here

for tp in [('Attack','Defense'), ('Sp. Atk','Sp. Def')]:
    res = stats.ttest_rel(pokemon[tp[0]],pokemon[tp[1]])
    print(tp)
    print('p-value:', float(res.pvalue))    

('Attack', 'Defense')
p-value: 1.7140303479358558e-05
('Sp. Atk', 'Sp. Def')
p-value: 0.3933685997548122


#### What conclusions can you make?

In [ ]:
# Your comment here

There is a significant difference between Attach and Defense value among all pokemon. No significant difference could be found between `Sp. Atk` and `Sp. Def`.